# Spark Interoperability with the Fleet Iceberg V3 Tables

This notebook queries the **Smart Fleet** Snowflake-managed Iceberg V3 tables from Apache Spark
through the Snowflake **Horizon Iceberg REST catalog**, with **access controls enforced by
Snowflake** (the same masking policies apply cross-engine).

It demonstrates:
- Reading the fleet tables created by the `001`–`010` SQL pipeline via the REST catalog
- `variant_get` over the `TELEMETRY_DATA` VARIANT column
- **Enforced governance**: the same `VEHICLE_REGISTRY` query returns full PII as the engineer
  role and *masked* PII as `FLEET_ANALYST` — enforced by Horizon, not by Spark

## Prerequisites
1. Completed the Snowflake setup (`task demo-up`) so the fleet tables and masking policies exist
2. Apache **Spark 4.0+** (required for VARIANT support) and **Java 17+**
3. A named `snow` CLI connection configured with **key-pair auth** (no PAT required) —
   this notebook mints a short-lived JWT with `snow connection generate-jwt`
4. Config is read from environment variables exported from `.env/iceberg.env`

## Configuration

All values come from `.env/iceberg.env` (exported into this process by the `task` runner).
Authentication uses a key-pair JWT minted from the named `snow` CLI connection — the same
mechanism the streaming pipeline uses. No Personal Access Token is involved.

In [ ]:
import os
import subprocess
import time
from contextlib import contextmanager
from datetime import datetime


@contextmanager
def step(label):
    """Print clear START / DONE / FAILED markers with elapsed time so you can
    tell from the Jupyter output when a cell started and whether it finished.
    """
    start = time.perf_counter()
    print(f"▶ START  {label}  [{datetime.now():%H:%M:%S}]", flush=True)
    try:
        yield
    except Exception as exc:
        elapsed = time.perf_counter() - start
        print(f"✖ FAILED {label}  after {elapsed:,.1f}s  ({type(exc).__name__}: {exc})", flush=True)
        raise
    else:
        elapsed = time.perf_counter() - start
        print(f"✔ DONE   {label}  in {elapsed:,.1f}s", flush=True)


# --- Horizon REST catalog ---
# Format: https://<account_identifier>.snowflakecomputing.com/polaris/api/catalog
horizon_catalog_uri = os.environ["SPARK_HORIZON_CATALOG_URI"]

# The Snowflake database is the Iceberg 'warehouse' (catalog) name.
catalog_name = os.environ["SPARK_CATALOG_NAME"]

# Named snow CLI connection used to mint the JWT (key-pair auth).
cli_connection = os.environ["CLI_CONNECTION_NAME"]

# Bronze schema (Iceberg namespace) holding the fleet tables.
bronze_schema = os.environ.get("DEMO_SCHEMA_NAME_BRONZE", "RAW")

# Roles for the enforced-masking demo.
analyst_role = os.environ.get("DEMO_ANALYST_ROLE_NAME", "FLEET_ANALYST")
engineer_role = os.environ.get("DEMO_ENGINEER_ROLE_NAME", "V3_DEMO_ICEBERG_ENGINEER_ROLE")

# Cloud provider selects the Iceberg cloud SDK bundle + FileIO. Even with
# Snowflake-managed storage, Spark reads data files through cloud SDK bundles,
# so match this to your Snowflake account's cloud (aws | gcp | azure).
cloud_provider = os.environ.get("SPARK_CLOUD_PROVIDER", "aws").lower()
aws_region = os.environ.get("AWS_REGION", "us-east-1")

# Iceberg runtime + cloud bundle version (Spark 4.0 / Scala 2.13).
ICEBERG_VER = os.environ.get("SPARK_ICEBERG_VERSION", "1.10.1")


def generate_jwt(connection: str) -> str:
    """Mint a key-pair JWT for the named snow CLI connection (valid ~60 min).

    Shells out to `snow connection generate-jwt`, which signs a JWT with the
    private key already configured in the connection. No PAT required.
    """
    result = subprocess.run(
        ["snow", "connection", "generate-jwt", "--connection", connection, "--silent"],
        capture_output=True,
        text=True,
        check=True,
    )
    token = result.stdout.strip()
    if not token:
        raise RuntimeError("snow connection generate-jwt returned empty output")
    return token


print(f"Catalog (database): {catalog_name}")
print(f"Catalog URI:        {horizon_catalog_uri}")
print(f"Cloud provider:     {cloud_provider}")
print(f"Iceberg version:    {ICEBERG_VER}")

In [ ]:
# Map cloud provider -> (Iceberg cloud bundle, FileIO implementation).
_CLOUD = {
    "aws": ("org.apache.iceberg:iceberg-aws-bundle", "org.apache.iceberg.aws.s3.S3FileIO"),
    "gcp": ("org.apache.iceberg:iceberg-gcp-bundle", "org.apache.iceberg.gcp.gcs.GCSFileIO"),
    "azure": ("org.apache.iceberg:iceberg-azure-bundle", "org.apache.iceberg.azure.adlsv2.ADLSFileIO"),
}
if cloud_provider not in _CLOUD:
    raise ValueError(f"SPARK_CLOUD_PROVIDER must be one of {list(_CLOUD)}; got {cloud_provider!r}")

bundle_pkg, file_io = _CLOUD[cloud_provider]
bundle = f"{bundle_pkg}:{ICEBERG_VER}"
print(f"Using bundle: {bundle}")
print(f"Using FileIO: {file_io}")

## Create a Spark session scoped to a Snowflake role

`build_spark(role)` creates a Spark session authenticated to the Horizon REST catalog as the
given role. Because Snowflake enforces access controls (including masking policies) based on
this role, switching roles changes what the **same** query returns — the enforcement happens
in Snowflake, not in Spark.

In [ ]:
from pyspark.sql import SparkSession
import findspark

findspark.init()

CATALOG = "horizoncatalog"


def build_spark(role: str) -> SparkSession:
    """Create a Spark session authenticated to Horizon REST as `role`.

    Uses a key-pair JWT (no PAT) for the Iceberg REST catalog. A fresh JWT is
    minted per session so it is well within its validity window.
    """
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()

    jwt = generate_jwt(cli_connection)

    builder = (
        SparkSession.builder
        .appName(f"horizon-iceberg-{role}")
        .master("local[*]")
        .config("spark.ui.showConsoleProgress", "false")
        .config(
            "spark.jars.packages",
            f"org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:{ICEBERG_VER},{bundle}",
        )
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.defaultCatalog", CATALOG)
        .config(f"spark.sql.catalog.{CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
        .config(f"spark.sql.catalog.{CATALOG}.type", "rest")
        .config(f"spark.sql.catalog.{CATALOG}.uri", horizon_catalog_uri)
        .config(f"spark.sql.catalog.{CATALOG}.warehouse", catalog_name)
        .config(f"spark.sql.catalog.{CATALOG}.header.X-Iceberg-Access-Delegation", "vended-credentials")
        # Key-pair JWT auth (no PAT): bearer token + Snowflake token-type header.
        .config(f"spark.sql.catalog.{CATALOG}.token", jwt)
        .config(f"spark.sql.catalog.{CATALOG}.header.X-Snowflake-Authorization-Token-Type", "KEYPAIR_JWT")
        # Role drives Horizon access control (masking is enforced per role).
        .config(f"spark.sql.catalog.{CATALOG}.scope", f"session:role:{role}")
        .config(f"spark.sql.catalog.{CATALOG}.io-impl", file_io)
        .config(f"spark.sql.catalog.{CATALOG}.file-io-impl", file_io)
        .config("spark.sql.iceberg.vectorization.enabled", "false")
    )
    if cloud_provider == "aws":
        builder = builder.config(f"spark.sql.catalog.{CATALOG}.client.region", aws_region)

    with step(f"Create Spark session (role={role}; first run downloads JARs)"):
        spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    return spark

## 1. Connect as the engineer role (full access) and list the fleet tables

In [ ]:
spark = build_spark(engineer_role)

with step("List fleet namespaces and tables"):
    spark.sql("SHOW NAMESPACES").show()
    spark.sql(f"SHOW TABLES IN {bronze_schema}").show(truncate=False)

## 2. Query the VARIANT telemetry column with `variant_get`

`VEHICLE_TELEMETRY_STREAM.TELEMETRY_DATA` is a VARIANT (Iceberg V3). Spark 4.0 reads it
natively and `variant_get` extracts nested fields by JSON path.

In [ ]:
with step("Query telemetry VARIANT with variant_get"):
    spark.sql(f"""
SELECT
    VEHICLE_ID,
    EVENT_TIMESTAMP,
    variant_get(TELEMETRY_DATA, '$.speed_mph', 'double')                 AS speed_mph,
    variant_get(TELEMETRY_DATA, '$.engine.rpm', 'int')                   AS engine_rpm,
    variant_get(TELEMETRY_DATA, '$.engine.fuel_level_pct', 'double')     AS fuel_level_pct,
    variant_get(TELEMETRY_DATA, '$.diagnostics.check_engine', 'boolean') AS check_engine,
    variant_get(TELEMETRY_DATA, '$.metadata.region', 'string')          AS region
FROM {bronze_schema}.VEHICLE_TELEMETRY_STREAM
ORDER BY EVENT_TIMESTAMP DESC
LIMIT 20
""").show(truncate=False)

## 3. Enforced governance — full PII as the engineer role

`VEHICLE_REGISTRY` carries driver PII (`DRIVER_NAME`, `DRIVER_EMAIL`, `DRIVER_PHONE`) protected
by Snowflake masking policies. The engineer role is exempt, so it sees the raw values — even
from Spark.

In [ ]:
with step("Read VEHICLE_REGISTRY PII as engineer role (full)"):
    spark.sql(f"""
SELECT VEHICLE_ID, DRIVER_NAME, DRIVER_EMAIL, DRIVER_PHONE
FROM {bronze_schema}.VEHICLE_REGISTRY
ORDER BY VEHICLE_ID
LIMIT 10
""").show(truncate=False)

## 4. Same query as `FLEET_ANALYST` — PII is masked

Reconnect as the analyst role and run the **identical** query. The masking policies are applied
by Horizon, so the PII columns come back masked. Nothing changed in Spark — only the role.

In [ ]:
spark = build_spark(analyst_role)

with step("Read VEHICLE_REGISTRY PII as FLEET_ANALYST (masked)"):
    spark.sql(f"""
SELECT VEHICLE_ID, DRIVER_NAME, DRIVER_EMAIL, DRIVER_PHONE
FROM {bronze_schema}.VEHICLE_REGISTRY
ORDER BY VEHICLE_ID
LIMIT 10
""").show(truncate=False)

## Summary

Apache Spark read the **same** Snowflake-managed Iceberg V3 fleet tables through the Horizon
REST catalog with vended credentials, queried VARIANT data with `variant_get`, and saw
Snowflake's masking policies **enforced consistently across engines** — full PII for the
engineer role, masked PII for `FLEET_ANALYST`. Governance lives with the data, not the engine.